# 01. Preprocessing — Playground Series S6E7 (Predicting Student Health Risk)

타겟: `health_condition` (at-risk / unhealthy / fit), 평가지표: Balanced Accuracy

이 노트북은 원본 `train.csv` / `test.csv`를 읽어서
1. 결측치 플래그 생성
2. 결측치 대치(imputation)
3. 범주형 인코딩(순서형/명목형 구분)
4. 타겟 라벨 인코딩
5. 전처리 결과를 `processed/` 폴더에 저장

까지 수행합니다.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

DATA_DIR = Path("../playground-series-s6e7")
OUT_DIR = DATA_DIR / "processed"
OUT_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print("train:", train.shape, " test:", test.shape)
train.head()

train: (690088, 15)  test: (295753, 14)


,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


## 1. 컬럼 정의

EDA에서 확인한 내용을 기준으로 컬럼 타입을 명시적으로 정의합니다.

In [2]:
TARGET = "health_condition"
ID_COL = "id"

NUMERIC_COLS = [
    "sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake",
]

# 순서형 범주형: 순서 정보를 살려서 ordinal encoding
ORDINAL_COLS = {
    "stress_level": ["low", "medium", "high"],
    "sleep_quality": ["poor", "average", "good"],
    "physical_activity_level": ["sedentary", "moderate", "active"],
    "smoking_alcohol": ["no", "occasional", "yes"],
}

# 명목형 범주형: 순서 없음 -> one-hot encoding
NOMINAL_COLS = ["diet_type", "gender"]

print("numeric:", NUMERIC_COLS)
print("ordinal:", list(ORDINAL_COLS.keys()))
print("nominal:", NOMINAL_COLS)

numeric: ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']
ordinal: ['stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol']
nominal: ['diet_type', 'gender']


## 2. 결측치 플래그 생성

모든 피처 컬럼에 결측이 존재하므로, 대치 전에 "결측 여부" 자체를 이진 피처로 남겨둡니다.
(결측 패턴이 타겟과 관련 있을 가능성을 열어둠 — EDA 단계에서 검증 예정)

In [3]:
FEATURE_COLS = NUMERIC_COLS + list(ORDINAL_COLS.keys()) + NOMINAL_COLS

for df in (train, test):
    for col in FEATURE_COLS:
        df[f"{col}_isnull"] = df[col].isna().astype(np.int8)

missing_flag_cols = [f"{c}_isnull" for c in FEATURE_COLS]
train[missing_flag_cols].sum().sort_values(ascending=False)

stress_level_isnull               82811
sleep_duration_isnull             75999
sleep_quality_isnull              58331
calorie_expenditure_isnull        52853
water_intake_isnull               43477
physical_activity_level_isnull    36621
smoking_alcohol_isnull            28582
gender_isnull                     21373
step_count_isnull                 13916
bmi_isnull                        13898
heart_rate_isnull                  7833
exercise_duration_isnull           6901
diet_type_isnull                   6901
dtype: int64

## 3. 결측치 대치

- 수치형: train 기준 median으로 대치 (이상치에 덜 민감)
- 범주형: `"missing"` 이라는 새 카테고리로 대치 (임의 최빈값 대치보다 정보 손실이 적음)

대치 기준값은 반드시 **train으로만 계산**해서 test에 동일하게 적용합니다 (data leakage 방지).

In [4]:
# 수치형: train median으로 대치
numeric_medians = train[NUMERIC_COLS].median()

for df in (train, test):
    for col in NUMERIC_COLS:
        df[col] = df[col].fillna(numeric_medians[col])

# 범주형(순서형 + 명목형): 'missing' 카테고리로 대치
categorical_cols = list(ORDINAL_COLS.keys()) + NOMINAL_COLS
for df in (train, test):
    for col in categorical_cols:
        df[col] = df[col].fillna("missing")

print("남은 결측치 (feature 컬럼 기준):")
print(train[FEATURE_COLS].isna().sum().sum(), test[FEATURE_COLS].isna().sum().sum())

남은 결측치 (feature 컬럼 기준):
0 0


## 4. 범주형 인코딩

- 순서형 컬럼: 정의된 순서대로 `OrdinalEncoder` 적용 (`missing`은 -1로 별도 처리)
- 명목형 컬럼: `pd.get_dummies`로 one-hot 인코딩

In [5]:
# 순서형 인코딩
for col, order in ORDINAL_COLS.items():
    categories = order + ["missing"]
    encoder = OrdinalEncoder(categories=[categories])
    train[col] = encoder.fit_transform(train[[col]])
    test[col] = encoder.transform(test[[col]])

# 명목형 원-핫 인코딩 (train/test 컬럼 정합성 맞추기)
train_ohe = pd.get_dummies(train[NOMINAL_COLS], prefix=NOMINAL_COLS)
test_ohe = pd.get_dummies(test[NOMINAL_COLS], prefix=NOMINAL_COLS)
test_ohe = test_ohe.reindex(columns=train_ohe.columns, fill_value=0)

train = pd.concat([train.drop(columns=NOMINAL_COLS), train_ohe], axis=1)
test = pd.concat([test.drop(columns=NOMINAL_COLS), test_ohe], axis=1)

train.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,stress_level,...,diet_type_isnull,gender_isnull,diet_type_balanced,diet_type_missing,diet_type_non-veg,diet_type_veg,gender_female,gender_male,gender_missing,gender_other
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,2.0,...,0,0,False,False,False,True,True,False,False,False
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,0.0,...,0,0,False,False,True,False,False,False,False,True
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,2.0,...,0,0,False,False,False,True,False,True,False,False
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,2.0,...,0,0,False,False,False,True,True,False,False,False
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,3.0,...,0,0,False,False,False,True,False,True,False,False


## 5. 타겟 라벨 인코딩

In [6]:
target_encoder = LabelEncoder()
train[TARGET] = target_encoder.fit_transform(train[TARGET])

print(dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_))))
train[TARGET].value_counts()

{'at-risk': np.int64(0), 'fit': np.int64(1), 'unhealthy': np.int64(2)}


health_condition
0    592561
2     57724
1     39803
Name: count, dtype: int64

## 6. 저장

전처리된 train/test를 `processed/` 폴더에 저장합니다. 모델링 노트북에서는 이 파일을 바로 불러오면 됩니다.

In [7]:
train.to_csv(OUT_DIR / "train_processed.csv", index=False)
test.to_csv(OUT_DIR / "test_processed.csv", index=False)

print("saved:", OUT_DIR / "train_processed.csv", train.shape)
print("saved:", OUT_DIR / "test_processed.csv", test.shape)

saved: ../playground-series-s6e7/processed/train_processed.csv (690088, 34)
saved: ../playground-series-s6e7/processed/test_processed.csv (295753, 33)
